In [ ]:
# MDS650_v260806_temperature_profile.ipynb
"""
Extracts the biweekly temperature series (mean, standard deviation, and
TEMPORAL variance, i.e., between days within each biweekly period -
not to be confused with the SPATIAL std we calculate in terrain_profile.py)
for a point, using ERA5-Land Daily Aggregates.

HOW TO READ THE RESULT
-----------------------
mean_C    Mean temperature for that biweekly period, in °C.
std_C     How much the temperature varied day to day within that biweekly period.
          High = biweekly period with very dissimilar days (e.g., mix of cool
          and hot days); low = stable temperature throughout the period.
var_C     The same variability, expressed as variance (std squared).
"""
import calendar
from datetime import date, datetime, timedelta
from pathlib import Path
import pandas as pd
import ee
# Initialize the Earth Engine API (must be called before any EE operations)
ee.Initialize()

from period_utils import build_biweekly_periods


def get_temperature_biweekly(lat, lon, start_date="2016-01-01", end_date=None):
    """
    Extracts biweekly temperature statistics from ERA5-Land Daily Aggregates
    for a given point.
    
    For each biweekly period, computes the mean, standard deviation, and variance
    of daily temperatures. Values are converted from Kelvin to Celsius.
    
    Args:
        lat: Latitude of the point in degrees
        lon: Longitude of the point in degrees
        start_date: Start date for the analysis period (str "YYYY-MM-DD")
        end_date: End date for the analysis period (str "YYYY-MM-DD");
                  None defaults to today.
                  NOTE: ERA5-Land has a few days of latency in its
                  publication -> the most recent biweekly periods may not
                  be available yet even though they have "already passed"
                  on the calendar.
    
    Returns:
        pandas.DataFrame: Biweekly temperature statistics with columns:
                          period_start, period_end, label,
                          mean_C, std_C, var_C
    """
    # Parse date strings to Python date objects
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date() if end_date else date.today()

    # Generate the list of complete biweekly periods within the date range
    periods = build_biweekly_periods(start, end)
    print(f"[DEBUG] {len(periods)} biweekly periods to process, from {start} to {end}")

    # Create Earth Engine point geometry for the extraction location
    point = ee.Geometry.Point([lon, lat])

    # Load ERA5-Land Daily Aggregates temperature collection
    # Filter to the date range (adding 1 day to end to include the last day)
    era5_coll = (
        ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
        .select('temperature_2m')  # 2-meter air temperature in Kelvin
        .filterDate(str(start), str(end + timedelta(days=1)))
    )

    # Create a combined reducer that computes mean, standard deviation, and variance
    # sharedInputs=True applies all reducers to the same input data
    combined_reducer = (
        ee.Reducer.mean()
        .combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True)
        .combine(reducer2=ee.Reducer.variance(), sharedInputs=True)
    )

    # Build the list of periods as an ee.List of dictionaries, to
    # process them ALL on the server with .map() -> a single network
    # call at the end, instead of one per biweekly period (240 before).
    # This is a major performance optimization.
    ee_periods = ee.List([
        {'label': label, 'start': str(p_start), 'end': str(p_end)}
        for label, p_start, p_end in periods
    ])

    def compute_period(period):
        """
        Server-side function to compute temperature statistics for a single period.
        This runs on Google Earth Engine servers, not locally.
        
        Args:
            period: ee.Dictionary with keys 'label', 'start', 'end'
        
        Returns:
            ee.Feature with computed temperature statistics (mean, stdDev, variance)
        """
        period = ee.Dictionary(period)
        p_start = ee.Date(period.get('start'))
        p_end = ee.Date(period.get('end'))

        # Filter the collection to only include images within this period
        filtered = era5_coll.filterDate(p_start, p_end)
        
        # Reduce the filtered collection to get temporal statistics
        # Then extract the pixel value at the point location
        stats = filtered.reduce(combined_reducer).reduceRegion(
            reducer=ee.Reducer.first(),  # Take the first (only) pixel value at the point
            geometry=point,
            scale=9000,  # ERA5-Land native resolution (~9 km)
            maxPixels=1e9
        )

        # Return as a Feature with computed statistics and period metadata
        return ee.Feature(
            None,
            stats
            .set('label', period.get('label'))
            .set('period_start', p_start.format('YYYY-MM-dd'))
            .set('period_end', p_end.advance(-1, 'day').format('YYYY-MM-dd'))
            # p_end is exclusive, so subtract 1 day to get the actual last day included
        )

    # Map the compute_period function over all periods (server-side execution)
    features = ee.FeatureCollection(ee_periods.map(compute_period))
    
    # Single network call to retrieve ALL periods' results at once
    result = features.getInfo()

    # Parse the Earth Engine response into a list of dictionaries
    rows = []
    for f in result['features']:
        props = f['properties']
        
        # Extract raw values from Earth Engine (in Kelvin)
        mean_k = props.get('temperature_2m_mean')
        std_k = props.get('temperature_2m_stdDev')
        var_k = props.get('temperature_2m_variance')

        # Convert Kelvin to Celsius and round, handling potential None values
        rows.append({
            'period_start': props.get('period_start'),
            'period_end': props.get('period_end'),
            'label': props.get('label'),
            # 'lat': lat,   # Uncomment if you want coordinates in the output
            # 'lon': lon,   # Uncomment if you want coordinates in the output
            'mean_C': round(mean_k - 273.15, 2) if mean_k is not None else None,
            'std_C': round(std_k, 2) if std_k is not None else None,
            'var_C': round(var_k, 2) if var_k is not None else None,
        })

    # Create DataFrame and sort chronologically by period start date
    df = pd.DataFrame(rows)
    df['period_start'] = pd.to_datetime(df['period_start'])
    df = df.sort_values('period_start').reset_index(drop=True)
    return df


def save_temperature_profile(df, out_prefix="temperature_biweekly", output_dir="../databases"):
    """
    Saves the temperature series with a timestamp in the filename:
    {out_prefix}-vYYMMDDHHMMSS.csv (same pattern as terrain_profile/soil_profile)
    
    Args:
        df: DataFrame from get_temperature_biweekly()
        out_prefix: Base name for the output CSV file
        output_dir: Directory where the CSV is saved (created if it doesn't exist)
    
    Returns:
        pathlib.Path: Path to the saved CSV file
    """
    # Generate a unique filename with timestamp
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    # Create output directory if it doesn't exist
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    # Save to CSV without the default pandas index
    df.to_csv(out_path, index=False)
    print(f"CSV saved to {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path


if __name__ == "__main__":
    
    # Test point: Sugarcane field in Queensland, Australia
    LAT = -19.689669877950884
    LON = 147.22717515914223
        
    # Reference points for quick access (commented out):
    # El Playon         --||     7.4584221918243045,    -73.222052853104
    # Finca Matanza     --||     7.300921,              -73.009794
    # Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
    # Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223

    # Extract biweekly temperature data from 2016 onwards
    df = get_temperature_biweekly(LAT, LON, start_date="2016-01-01")
    
    # Save the temperature profile to CSV
    out_path = save_temperature_profile(df)
    
    # Display the first 10 rows for inspection
    print(df.head(10))

[DEBUG] 254 biweekly periods to process, from 2016-01-01 to 2026-08-12
CSV saved to ../databases/temperature_biweekly-v260812003422.csv (254x6)
  period_start  period_end       label  mean_C  std_C  var_C
0   2016-01-01  2016-01-15  2016-01_Q1   28.03   1.30   1.69
1   2016-01-16  2016-01-31  2016-01_Q2   28.35   1.06   1.12
2   2016-02-01  2016-02-15  2016-02_Q1   28.11   1.01   1.02
3   2016-02-16  2016-02-29  2016-02_Q2   29.07   1.30   1.68
4   2016-03-01  2016-03-15  2016-03_Q1   26.32   1.23   1.52
5   2016-03-16  2016-03-31  2016-03_Q2   26.06   0.84   0.71
6   2016-04-01  2016-04-15  2016-04_Q1   25.46   0.61   0.37
7   2016-04-16  2016-04-30  2016-04_Q2   24.50   0.42   0.18
8   2016-05-01  2016-05-15  2016-05_Q1   24.42   1.32   1.75
9   2016-05-16  2016-05-31  2016-05_Q2   24.05   0.79   0.63
